# M3GNet--Linear Atomic Sum baseline verification

This notebook verifies the controlled comparison used to isolate the Transformer contribution:

- baseline: `M3GNet blocks -> Linear atomic head -> Atomic Sum`
- target: `M3GNet blocks -> Transformer -> Linear atomic head -> Atomic Sum`

The architecture checks use assertions so that an unexpected code path stops the notebook immediately.

In [1]:
from pathlib import Path
import json
import random
import subprocess
import sys

REPO = Path.cwd().resolve()
SRC = REPO / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

branch = subprocess.run(
    ['git', '-C', str(REPO), 'branch', '--show-current'],
    check=True, capture_output=True, text=True
).stdout.strip()
print('Repository:', REPO)
print('Branch:', branch)
assert branch == 'm3gnet-linear-baseline'

Repository: D:\M3GNET+LINEAR
Branch: m3gnet-linear-baseline


## 1. Construct the two models with matched settings

Both models are initialised with the same seed. Their backbone parameters must therefore have identical names, shapes, and initial values.

In [2]:
import torch
from matgl.config import DEFAULT_ELEMENTS
from matgl.layers._readout_torch import LinearAtomicReadOut, TransformerAtomicReadOut
from matgl.models._m3gnet import M3GNet

COMMON = dict(
    element_types=DEFAULT_ELEMENTS,
    is_intensive=False,
    cutoff=5.0,
    nblocks=3,
    dim_node_embedding=64,
    dim_edge_embedding=64,
    units=64,
)

torch.manual_seed(42)
baseline = M3GNet(**COMMON, readout_type='linear_atomic_sum')
torch.manual_seed(42)
target = M3GNet(
    **COMMON,
    readout_type='transformer',
    transformer_nhead=4,
    transformer_num_layers=1,
    transformer_dim_ff=128,
    transformer_dropout=0.0,
)

assert isinstance(baseline.final_layer, LinearAtomicReadOut)
assert isinstance(target.final_layer, TransformerAtomicReadOut)
assert not any(isinstance(module, torch.nn.TransformerEncoder) for module in baseline.modules())
assert any(isinstance(module, torch.nn.TransformerEncoder) for module in target.modules())
assert baseline.final_layer.atomic_head.in_features == target.final_layer.atomic_head.in_features == 64
assert baseline.final_layer.atomic_head.out_features == target.final_layer.atomic_head.out_features == 1

print('Baseline final layer:', baseline.final_layer)
print('Target final layer:', target.final_layer)
print('Baseline parameters:', sum(p.numel() for p in baseline.parameters()))
print('Target parameters:', sum(p.numel() for p in target.parameters()))

W0827 17:07:42.205000 32580 site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Baseline final layer: LinearAtomicReadOut(
  (atomic_head): Linear(in_features=64, out_features=1, bias=True)
)
Target final layer: TransformerAtomicReadOut(
  (transformer): TransformerEncoder(
    (layers): ModuleList(
      (0): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.0, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.0, inplace=False)
        (dropout2): Dropout(p=0.0, inplace=False)
      )
    )
  )
  (atomic_head): Linear(in_features=64, out_features=1, bias=True)
)
Baseline parameters: 263132
Target parameters: 296604


In [3]:
def backbone_state(model):
    return {
        name: value.detach().clone()
        for name, value in model.state_dict().items()
        if not name.startswith('final_layer.')
    }

baseline_backbone = backbone_state(baseline)
target_backbone = backbone_state(target)
assert baseline_backbone.keys() == target_backbone.keys()
assert all(
    baseline_backbone[name].shape == target_backbone[name].shape
    for name in baseline_backbone
)
assert all(
    torch.equal(baseline_backbone[name], target_backbone[name])
    for name in baseline_backbone
)
print(f'Backbone check passed for {len(baseline_backbone)} tensors.')
print('The only model-state differences are under final_layer.*')

Backbone check passed for 95 tensors.
The only model-state differences are under final_layer.*


## 2. Verify the baseline forward path

The hook below captures the tensor entering the linear readout. It must be exactly the node-feature tensor produced by the final M3GNet graph block. The final energy must equal the sum of the predicted atomic contributions.

In [4]:
from pymatgen.core import Lattice, Structure
from matgl.ext.pymatgen import Structure2Graph
from matgl.graph._compute import compute_pair_vector_and_distance

structure = Structure(
    Lattice.cubic(4.0),
    ['Mo', 'S'],
    [[0.0, 0.0, 0.0], [0.5, 0.5, 0.5]],
)
converter = Structure2Graph(element_types=DEFAULT_ELEMENTS, cutoff=5.0)
graph, lattice, state = converter.get_graph(structure)
graph.pbc_offshift = torch.matmul(graph.pbc_offset, lattice[0])
graph.pos = graph.frac_coords @ lattice[0]
graph.bond_vec, graph.bond_dist = compute_pair_vector_and_distance(
    graph.pos, graph.edge_index, graph.pbc_offshift
)

captured = {}
def capture_linear_input(_module, args):
    captured['linear_input'] = args[0].detach().clone()

hook = baseline.final_layer.register_forward_pre_hook(capture_linear_input)
energy = baseline(graph)
hook.remove()

last_graph_features = baseline.feature_dict['gc_3']['node_feat'].detach()
atomic_energies = baseline.feature_dict['readout'].view(-1)
assert torch.equal(captured['linear_input'], last_graph_features)
assert torch.allclose(energy.view(-1), atomic_energies.sum().view(-1))
print('Last M3GNet node features:', tuple(last_graph_features.shape))
print('Linear atomic contributions:', tuple(atomic_energies.shape))
print('Total energy:', energy.item())
print('Forward-path assertion: PASS')

Last M3GNet node features: (2, 64)
Linear atomic contributions: (2,)
Total energy: 0.5553873181343079
Forward-path assertion: PASS


In [5]:
from matgl.apps.pes import Potential

potential = Potential(model=baseline, calc_forces=True, calc_stresses=True)
energy, forces, stresses, _ = potential(graph, lattice[0], state)
loss = energy.sum() + forces.square().mean() + stresses.square().mean()
loss.backward()

assert torch.isfinite(energy).all()
assert torch.isfinite(forces).all()
assert torch.isfinite(stresses).all()
assert all(
    parameter.grad is None or torch.isfinite(parameter.grad).all()
    for parameter in baseline.parameters()
)
print('Energy shape:', tuple(energy.shape))
print('Force shape:', tuple(forces.shape))
print('Stress shape:', tuple(stresses.shape))
print('Energy/force/stress backward check: PASS')

Energy shape: ()
Force shape: (2, 3)
Stress shape: (3, 3)
Energy/force/stress backward check: PASS


## 3. Optional small-subset training

This is only a smoke experiment. Its errors must not be reported as scientific results. It checks that both architectures train on the same records and produce checkpoints and test metrics. Set `RUN_TRAINING = True` when ready.

In [6]:
SOURCE_DATA = Path(r'D:\M3GNET_Transformer\matglformer\MatPES-PBE-2025.2-random-2000-seed-2032.json')
SMOKE_ROOT = REPO / 'runs' / 'm3gnet_transformer_matched_smoke'
SUBSET_DATA = SMOKE_ROOT / 'matpes_smoke_128_seed42.json'
SMOKE_ROOT.mkdir(parents=True, exist_ok=True)

records = json.loads(SOURCE_DATA.read_text(encoding='utf-8'))
indices = random.Random(42).sample(range(len(records)), 128)
subset = [records[index] for index in indices]
SUBSET_DATA.write_text(json.dumps(subset), encoding='utf-8')
print('Subset:', SUBSET_DATA)
print('Structures:', len(subset))

Subset: D:\M3GNET+LINEAR\runs\m3gnet_transformer_matched_smoke\matpes_smoke_128_seed42.json
Structures: 128


In [9]:
RUN_TRAINING = True

common_args = [
    '--data', str(SUBSET_DATA),
    '--max-epochs', '2',
    '--batch-size', '8',
    '--accumulate-grad-batches', '1',
    '--num-workers', '0',
    '--accelerator', 'cpu',
    '--devices', '1',
    '--seed', '42',
]

commands = {
    'baseline': [
        sys.executable, str(REPO / 'train_m3gnet_linear_atomic_sum_full_matpes.py'),
        '--output-dir', str(SMOKE_ROOT / 'baseline'), *common_args,
    ],
    'transformer': [
        sys.executable, str(REPO / 'train_transformer_atomic_sum_full_matpes.py'),
        '--output-dir', str(SMOKE_ROOT / 'transformer'), *common_args,
        '--transformer-nhead', '4',
        '--transformer-num-layers', '1',
        '--transformer-dim-ff', '128',
        '--transformer-dropout', '0.0',
    ],
}

if RUN_TRAINING:
    for name, command in commands.items():
        print(f'Running {name}:')
        print(' '.join(command))
        subprocess.run(command, cwd=REPO, check=True)
else:
    print('Set RUN_TRAINING = True to run both two-epoch smoke experiments.')

Running baseline:
d:\anaconda\python.exe D:\M3GNET+LINEAR\train_m3gnet_linear_atomic_sum_full_matpes.py --output-dir D:\M3GNET+LINEAR\runs\m3gnet_transformer_matched_smoke\baseline --data D:\M3GNET+LINEAR\runs\m3gnet_transformer_matched_smoke\matpes_smoke_128_seed42.json --max-epochs 2 --batch-size 8 --accumulate-grad-batches 1 --num-workers 0 --accelerator cpu --devices 1 --seed 42
Running transformer:
d:\anaconda\python.exe D:\M3GNET+LINEAR\train_transformer_atomic_sum_full_matpes.py --output-dir D:\M3GNET+LINEAR\runs\m3gnet_transformer_matched_smoke\transformer --data D:\M3GNET+LINEAR\runs\m3gnet_transformer_matched_smoke\matpes_smoke_128_seed42.json --max-epochs 2 --batch-size 8 --accumulate-grad-batches 1 --num-workers 0 --accelerator cpu --devices 1 --seed 42 --transformer-nhead 4 --transformer-num-layers 1 --transformer-dim-ff 128 --transformer-dropout 0.0


In [10]:
results = {}
for name in ('baseline', 'transformer'):
    result_path = SMOKE_ROOT / name / 'test_results.json'
    if result_path.is_file():
        payload = json.loads(result_path.read_text(encoding='utf-8'))
        results[name] = {
            'best_val_total_loss': payload['best_val_total_loss'],
            **payload['test'],
        }
results

{'baseline': {'best_val_total_loss': 4.457891464233398,
  'test_Total_Loss': 3.683419942855835,
  'test_Energy_MAE': 2.361670970916748,
  'test_Force_MAE': 0.623529851436615,
  'test_Stress_MAE': 14.985066413879395,
  'test_Magmom_MAE': 0.0,
  'test_Charge_MAE': 0.0,
  'test_Energy_RMSE': 2.581360101699829,
  'test_Force_RMSE': 0.9500717520713806,
  'test_Stress_RMSE': 26.608339309692383,
  'test_Magmom_RMSE': 0.0,
  'test_Charge_RMSE': 0.0},
 'transformer': {'best_val_total_loss': 1.5626107454299927,
  'test_Total_Loss': 1.1584088802337646,
  'test_Energy_MAE': 1.136103630065918,
  'test_Force_MAE': 0.26946499943733215,
  'test_Stress_MAE': 3.7695209980010986,
  'test_Magmom_MAE': 0.0,
  'test_Charge_MAE': 0.0,
  'test_Energy_RMSE': 1.5215651988983154,
  'test_Force_RMSE': 0.5161048769950867,
  'test_Stress_RMSE': 7.67513370513916,
  'test_Magmom_RMSE': 0.0,
  'test_Charge_RMSE': 0.0}}